# 🔒 Private LLM Harness for Google Colab

Run open-weight models from Hugging Face **on the Colab machine's own GPU**. Your text and files are processed on that machine and are never sent to an AI API (OpenAI, Anthropic, Google Gemini, Hugging Face Inference…).

```
Hugging Face ──(model weights only)──▶  Colab VM + GPU  ◀── your files
                                          model, data, logs stay here
                                          deleted when you delete the runtime
```

### Quick start
1. **Runtime → Change runtime type → T4 GPU** (free) or L4/A100 (paid). TPU works too but is slow and experimental.
2. Create a **Read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). In Colab click 🔑 **Secrets** in the left bar, add `HF_TOKEN` and switch on *Notebook access*. The token is never written into the notebook.
3. Run the cells **in order** (▶ or Shift+Enter). Each one ends with ✅ when it worked. If a cell fails, fix it and rerun just that cell.

### Before you use real sensitive data
* **Edit → Notebook settings → tick "Omit code cell output when saving this notebook".** Otherwise printed answers get saved into the notebook file in your Google Drive.
* **This removes third-party AI services from your data's path. It doesn't make Colab itself approved for your data.** Colab runs on Google Cloud. Check that your organisation allows Colab for this kind of data. Consumer Colab isn't covered by agreements such as a HIPAA BAA. For regulated data, use **Colab Enterprise** or your own GPU server. The code works on any Jupyter machine with a GPU.
* Test everything with the built-in **synthetic data** first.

| Level | What you get | Who it's for |
|---|---|---|
| 0 | Hardware check, install, login | everyone |
| 1 | Chat loop in 5 lines | beginners: see it work |
| 2 | Memory, system prompt, streaming | learn how chat models work |
| 3 | Any HF model, auto-sized to your GPU | choose quality vs speed |
| 4 | Network kill-switch, PII redaction, audit log | the compliance layer |
| 5 | Questions about your own files (RAG) | your documents |
| 6 | One `Harness` for chat, extraction and batch jobs | real work |
| 7 | Chat app inside the notebook | daily use |
| 8 | Wipe everything | always, at the end |

> 💡 Levels 1, 2 and 5 wait for your input. **Press Enter on an empty box to continue** (or click *Cancel* on the upload box).

## Level 0 · Setup
**0.1** checks which accelerator you have and installs what's missing (Colab already ships PyTorch). It also turns off library telemetry and creates a private working folder on the VM's temporary disk. Google Drive is deliberately **not** mounted.

In [ ]:
#@title 0.1 · Check hardware & install (≈1 min)
import os, sys, glob, shutil, subprocess
os.environ.update(HF_HUB_DISABLE_TELEMETRY="1", DO_NOT_TRACK="1", TOKENIZERS_PARALLELISM="false")

def pip(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

if shutil.which("nvidia-smi"):
    ACCEL = "gpu"
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
elif os.environ.get("TPU_ACCELERATOR_TYPE") or os.environ.get("COLAB_TPU_ADDR") or glob.glob("/dev/accel*") or glob.glob("/dev/vfio/[0-9]*"):
    ACCEL = "tpu"
else:
    ACCEL = "cpu"

pip("transformers>=4.56", "accelerate", "sentence-transformers", "pypdf", "python-docx", *(["bitsandbytes"] if ACCEL == "gpu" else []))
if ACCEL == "tpu":
    import torch
    try: import torch_xla
    except ImportError:
        try: pip(f"torch_xla[tpu]=={torch.__version__.split('+')[0]}")
        except subprocess.CalledProcessError: ACCEL = "cpu"; print("⚠️ Couldn't install torch_xla. Falling back to CPU.")

WORKDIR = "/content/private" if os.path.isdir("/content") else os.path.abspath("private")
os.makedirs(WORKDIR, exist_ok=True)
print(f"✅ Accelerator: {ACCEL.upper()} | private workspace: {WORKDIR}")
if ACCEL == "cpu": print("⚠️ No GPU found. Runtime → Change runtime type → T4 GPU, then run this cell again.")

**0.2** logs in to Hugging Face with your secret, so you can download gated models such as Llama. **The token only downloads model weights. Your data never goes to Hugging Face.** Public models (like the Qwen defaults here) work without a token.

In [ ]:
#@title 0.2 · Hugging Face login (token stays in Colab Secrets)
from huggingface_hub import login, whoami
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:  # not in Colab, or secret missing / access not granted
    import getpass
    print(f"Colab secret unavailable ({type(e).__name__}). Paste a token, or press Enter to skip.")
    HF_TOKEN = getpass.getpass("HF token: ").strip()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in as", whoami()["name"])
else:
    print("✅ No token: public models only")
del HF_TOKEN

## Level 1 · The simplest chat loop
A `pipeline` downloads a small model (0.5B parameters, ~1 GB) and wraps it in one function call. Type a question, get an answer, press Enter on an empty line to stop.

**Try:** "Explain GDPR in two sentences." **Notice:** it has no memory. Ask "what did I just ask?" and it won't know. Level 2 fixes that.

In [ ]:
#@title Level 1 · Simplest possible chat loop
from transformers import pipeline
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

while (q := input("\nYou (Enter = stop): ").strip()):
    reply = pipe([{"role": "user", "content": q}], max_new_tokens=256)[0]["generated_text"][-1]["content"]
    print("AI:", reply)

## Level 2 · Memory, a system prompt and streaming
How chat models actually work:
* **The model has no memory.** Each turn, we resend the whole conversation (`history`) as a list of `{"role", "content"}` messages.
* The **system prompt** sets behaviour for the whole conversation.
* The **chat template** turns messages into the exact token format the model was trained on. Using the wrong format is the #1 cause of rambling output.
* `temperature` sets randomness: 0 gives the same answer every time (good for facts and extraction), ~0.7 is more creative.
* A **streamer** prints tokens as they're generated.

In [ ]:
#@title Level 2 · Chat with memory + streaming
import torch
from transformers import TextStreamer
model, tok = pipe.model, pipe.tokenizer   # a pipeline is just a model + a tokenizer

def chat(messages, max_new_tokens=512, temperature=0.7):
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.9) if temperature > 0 else dict(do_sample=False)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, streamer=TextStreamer(tok, skip_prompt=True, skip_special_tokens=True), **sampling)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

history = [{"role": "system", "content": "You are a concise, careful assistant. Say so when you are unsure."}]
while (q := input("\nYou (Enter = stop): ").strip()):
    history.append({"role": "user", "content": q})
    print("AI: ", end="")
    history.append({"role": "assistant", "content": chat(history)})

## Level 3 · Pick any Hugging Face model
Use the dropdown or paste any chat model ID from huggingface.co (look for *Instruct*, *-it* or *chat* in the name). The loader:
* **Checks access before downloading** and gives a clear error for typos or gated models. For gated models (Llama, Mistral), accept the licence on the model's page first, using the same account as your token.
* **Auto-sizes the model.** If it won't fit in GPU memory at 16-bit, it's loaded in 4-bit (NF4), which needs ~4× less memory for a small loss in quality. On a free T4 (15 GB), models up to ~14B fit in 4-bit.
* **Security defaults:** `use_safetensors=True` refuses pickle files, which can run code. `trust_remote_code=False` won't run Python shipped inside a model repo.
* **Pins the exact revision** (commit SHA), so you can show which model version produced which output. Paste a SHA into `REVISION` to reproduce a past run.

| GPU | Rough guide |
|---|---|
| T4 (free, 15 GB) | 1.5–3B full precision; 7–8B in 4-bit |
| L4 (24 GB) / A100 (40–80 GB) | 7–8B full precision; 14–32B in 4-bit |
| TPU / CPU | ≤3B, slow. TPU support is experimental: the first tokens take minutes while it compiles. |

In [ ]:
#@title Level 3 · Load a model (auto-quantised to fit)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # @param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-14B-Instruct", "microsoft/Phi-4-mini-instruct", "meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3"] {allow-input: true}
QUANTIZE = "auto"  # @param ["auto", "4bit", "8bit", "none"]
REVISION = ""  # @param {type:"string"}
import gc, torch, contextlib
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_model(model_id, quantize="auto", revision=""):
    try: info = HfApi().model_info(model_id, revision=revision or None)
    except Exception as e:
        raise RuntimeError(f"Can't access {model_id}: {e}\n→ Check the spelling. For gated models, accept the licence at "
                           f"https://huggingface.co/{model_id} and make sure your HF_TOKEN belongs to that account.") from None
    n_params = info.safetensors.total if info.safetensors else None
    quant = None
    if ACCEL == "gpu":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        if quantize == "auto":  # 16-bit needs ~2 bytes/param plus ~30% for activations & cache
            quantize = "none" if not n_params or n_params * 2 * 1.3 < torch.cuda.mem_get_info()[0] else "4bit"
        quant = {"4bit": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True),
                 "8bit": BitsAndBytesConfig(load_in_8bit=True)}.get(quantize)
    else:
        dtype, quantize = torch.bfloat16, "none"  # bitsandbytes needs a GPU
    tok = AutoTokenizer.from_pretrained(model_id, revision=info.sha)
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=info.sha, dtype=dtype, quantization_config=quant,
                                                 device_map="auto" if ACCEL == "gpu" else None,
                                                 use_safetensors=True, trust_remote_code=False).eval()
    if ACCEL == "tpu":
        import torch_xla.core.xla_model as xm
        model.to(xm.xla_device()); model.generation_config.cache_implementation = "static"  # fixed shapes = fewer recompiles
    tok.pad_token = tok.pad_token or tok.eos_token
    meta = dict(id=model_id, revision=info.sha, quantize=quantize, params_b=round(n_params / 1e9, 1) if n_params else None)
    return model, tok, meta

model = pipe = None; gc.collect(); torch.cuda.empty_cache()   # free the Level 1 model first
with network_allowed() if "network_allowed" in globals() else contextlib.nullcontext():  # works before & after Level 4
    model, tok, META = load_model(MODEL_ID, QUANTIZE, REVISION)
mem = f" | GPU memory used {torch.cuda.memory_allocated() / 1e9:.1f} GB" if ACCEL == "gpu" else ""
print(f"✅ {META}{mem}\nAI: ", end="")
_ = chat([{"role": "user", "content": "In one sentence: what are you?"}], temperature=0)

## Level 4 · Privacy guardrails
The model already runs locally. These three layers protect you from **accidents**:

1. **Network kill-switch.** Once models are downloaded, this Python process can't open any outside connection, so no library, plugin or typo can upload your data. Downloads only happen inside an explicit `with network_allowed():` block. *Limit: it covers this notebook's Python process, not shell commands you run with `!`.*
2. **PII redaction.** Emails, phone numbers, cards, IBANs, SSNs, UK NI numbers, postcodes and IPs are matched with regular expressions. Names are found by a small local NER model. Each value is swapped for a placeholder such as `[EMAIL_1]` **before the model sees it**, and swapped back in the answer. The placeholder ↔ value map stays in memory only. This keeps PII out of prompts, logs and the model's context (data minimisation). **Redaction is never perfect**, so add patterns for your own identifiers (patient IDs, case numbers…) to `PII_PATTERNS`.
3. **Audit log.** Each model call is appended to `audit.jsonl`: time, model, revision, prompt hash, token counts and duration. Prompt text is **not** stored unless you tick `LOG_CONTENT`. This gives you evidence of what was processed and how, without making the log itself sensitive.

In [ ]:
#@title Level 4 · Kill-switch, redaction, audit log
USE_NER = True  # @param {type:"boolean"}
NER_LABELS = "PER"  # @param ["PER", "PER,ORG", "PER,ORG,LOC"]
LOG_CONTENT = False  # @param {type:"boolean"}
import re, json, socket, hashlib, datetime, ipaddress, contextlib
import huggingface_hub.constants as hf_constants

# 1 · Network kill-switch: patch socket connects to allow loopback only (but not a local proxy, which would forward out)
from urllib.parse import urlparse
_real = globals().get("_real") or {"connect": socket.socket.connect, "connect_ex": socket.socket.connect_ex}  # rerun-safe
def _guarded(name):
    proxy_ports = {urlparse(v).port for k, v in os.environ.items() if k.lower().endswith("_proxy") and "://" in v}
    def connect(self, addr):
        if isinstance(addr, tuple):  # (host, port[, ...]); unix sockets are str and stay allowed
            try: local = ipaddress.ip_address(str(addr[0]).split("%")[0]).is_loopback
            except ValueError: local = addr[0] == "localhost"
            if not local or addr[1] in proxy_ports: raise ConnectionRefusedError(f"🔒 Blocked by privacy kill-switch: {addr[0]}. Wrap intentional downloads in `with network_allowed():`")
        return _real[name](self, addr)
    return connect

def set_network(enabled: bool):
    for name in _real: setattr(socket.socket, name, _real[name] if enabled else _guarded(name))
    hf_constants.HF_HUB_OFFLINE = not enabled          # Hugging Face libs use local cache only
    os.environ["HF_HUB_OFFLINE"] = "0" if enabled else "1"

@contextlib.contextmanager
def network_allowed():
    set_network(True)
    try: yield
    finally: set_network(False)

# 2 · PII redaction (order matters: specific patterns before general ones)
PII_PATTERNS = {
    "EMAIL": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "IBAN": r"\b[A-Z]{2}\d{2}(?: ?[A-Z0-9]{4}){3,7}(?: ?[A-Z0-9]{1,3})?\b",
    "CARD": r"\b(?:\d[ -]?){12,15}\d\b",
    "SSN": r"\b\d{3}-\d{2}-\d{4}\b",
    "NI_NUMBER": r"\b[A-CEGHJ-PR-TW-Z]{2} ?\d{2} ?\d{2} ?\d{2} ?[A-D]\b",
    "IP": r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
    "PHONE": r"(?<![\w-])\+?\(?\d(?:[\s().-]{0,2}\d){8,14}(?!\w)",   # 9+ digits, so dates aren't caught
    "POSTCODE": r"\b[A-Z]{1,2}\d[A-Z\d]? ?\d[A-Z]{2}\b",
}

class Redactor:
    def __init__(self, patterns=PII_PATTERNS, ner=None, ner_labels=("PER",)):
        self.patterns, self.ner, self.labels = patterns, ner, ner_labels
        self.vault, self._rev, self._n = {}, {}, {}      # vault: placeholder → original (RAM only)
    def _tag(self, kind, value):
        if value not in self._rev:
            self._n[kind] = self._n.get(kind, 0) + 1
            tag = f"[{kind}_{self._n[kind]}]"
            self.vault[tag], self._rev[value] = value, tag
        return self._rev[value]
    def redact(self, text):
        for kind, rx in self.patterns.items():
            text = re.sub(rx, lambda m: self._tag(kind, m.group()), text)
        if self.ner:
            taken = [m.span() for m in re.finditer(r"\[[A-Z_]+_\d+\]", text)]  # never re-tag a placeholder
            for e in sorted(self.ner(text), key=lambda e: -e["start"]):  # right-to-left keeps offsets valid
                if e["entity_group"] in self.labels and e["score"] > 0.8 and not any(a < e["end"] and e["start"] < b for a, b in taken):
                    text = text[:e["start"]] + self._tag(e["entity_group"], text[e["start"]:e["end"]]) + text[e["end"]:]
        return text
    def restore(self, text):
        return re.sub(r"\[[A-Z_]+_\d+\]", lambda m: self.vault.get(m.group(), m.group()), text)

# 3 · Audit log: metadata + hashes, never raw text unless LOG_CONTENT
class AuditLog:
    def __init__(self, path, log_content=False): self.path, self.log_content = path, log_content
    def __call__(self, event, prompt="", output="", **meta):
        rec = {"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), "event": event,
               "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest()[:16], **meta}
        if self.log_content: rec.update(prompt=prompt, output=output)
        with open(self.path, "a") as f: f.write(json.dumps(rec) + "\n")

ner = None
if USE_NER:
    with network_allowed():
        ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple", stride=128, device="cpu")
redactor = Redactor(ner=ner, ner_labels=tuple(NER_LABELS.split(",")))
audit = AuditLog(f"{WORKDIR}/audit.jsonl", LOG_CONTENT)
set_network(False)  # 🔒 from here on, nothing leaves this process

# Self-tests
sample = "Hi, I'm John Smith (john.smith@acme.com, +44 20 7946 0958). Card 4111 1111 1111 1111, born 1990-01-15."
redacted = redactor.redact(sample)
print("Redacted:", redacted)
assert redactor.restore(redacted) == sample and "john.smith@" not in redacted and "1990-01-15" in redacted
try: socket.create_connection(("1.1.1.1", 443), timeout=5); print("⚠️ Network is still open!")
except OSError as e: print("✅ Redaction round-trips. Network:", e)

## Level 5 · Ask questions about your own documents (local RAG)
**Retrieval-Augmented Generation:** documents are split into overlapping chunks. Each chunk is turned into a vector by a small **local** embedding model. For each question, the closest chunks are pasted into the prompt as numbered sources the model must cite. Everything stays in memory on the VM: no vector database service, no cloud.

* Supports `.txt .md .csv .json .pdf .docx`. Scanned PDFs (images) have no text layer and need OCR first.
* Files are uploaded straight from your browser to `WORKDIR/docs` on the VM. You can also drag files into that folder using the 📁 panel.
* Click **Cancel** in the upload box to use a synthetic sample handbook.
* With `REDACT_DOCS`, PII is removed **before** indexing, so neither the index nor the model ever holds it.

In [ ]:
#@title Level 5 · Index documents & ask questions
EMBED_MODEL = "BAAI/bge-small-en-v1.5"  # @param ["BAAI/bge-small-en-v1.5", "BAAI/bge-m3", "sentence-transformers/all-MiniLM-L6-v2"] {allow-input: true}
UPLOAD = True  # @param {type:"boolean"}
REDACT_DOCS = True  # @param {type:"boolean"}
from pathlib import Path
from sentence_transformers import SentenceTransformer

class DocIndex:
    def __init__(self, embed_id, redactor=None, size=1000, overlap=150):
        with network_allowed():  # one-off download of the embedding model
            self.emb = SentenceTransformer(embed_id, device="cuda" if ACCEL == "gpu" else "cpu")
        self.redactor, self.size, self.overlap, self.chunks, self.vecs = redactor, size, overlap, [], None
    def add(self, name, text):
        if self.redactor: text = self.redactor.redact(text)  # redact whole doc so no PII is split across chunks
        new = [text[i:i + self.size] for i in range(0, max(len(text) - self.overlap, 1), self.size - self.overlap)]
        v = self.emb.encode(new, normalize_embeddings=True, convert_to_tensor=True)
        self.chunks += [(name, c) for c in new]
        self.vecs = v if self.vecs is None else torch.cat([self.vecs, v])
    def search(self, query, k=4):
        if not self.chunks: return []
        top = (self.vecs @ self.emb.encode(query, normalize_embeddings=True, convert_to_tensor=True)).topk(min(k, len(self.chunks)))
        return [(*self.chunks[i], s) for s, i in zip(top.values.tolist(), top.indices.tolist())]

def read_file(p: Path) -> str:
    if p.suffix.lower() == ".pdf":
        from pypdf import PdfReader; return "\n".join(page.extract_text() or "" for page in PdfReader(p).pages)
    if p.suffix.lower() == ".docx":
        import docx; return "\n".join(par.text for par in docx.Document(p).paragraphs)
    return p.read_text(errors="ignore")

DOCS = Path(WORKDIR, "docs"); DOCS.mkdir(exist_ok=True)
if UPLOAD:
    try:
        from google.colab import files
        with contextlib.chdir(DOCS): files.upload()   # saves straight into DOCS
    except ImportError: print(f"Not in Colab: copy files into {DOCS}")
if not any(DOCS.iterdir()):
    (DOCS / "SYNTHETIC_handbook.txt").write_text(
        "ACME Ltd Staff Handbook (synthetic test data)\n"
        "Annual leave: full-time staff get 25 days plus bank holidays. Book leave with HR at hr@acme.example or +44 20 7946 0000.\n"
        "Sick leave: tell your line manager, Priya Patel, before 10am. You can self-certify for up to 7 days.\n"
        "Expenses: claim within 30 days. Claims over £500 need approval from the finance director, Tom Baker.\n"
        "Data protection: never email customer data outside the company; report incidents to dpo@acme.example within 24 hours.\n")

index = DocIndex(EMBED_MODEL, redactor if REDACT_DOCS else None)
for p in sorted(DOCS.iterdir()):
    if p.is_file(): index.add(p.name, read_file(p)); print("📄", p.name)
print(f"✅ {len(index.chunks)} chunks indexed")

RAG_RULES = ("Answer ONLY from the numbered sources and cite them like [1]. If they don't contain the answer, say so. "
             "Keep placeholders such as [PER_1] exactly as written.")
def ask(question, k=4):
    q = redactor.redact(question)
    sources = "\n\n".join(f"[{i + 1}] ({name}) {chunk}" for i, (name, chunk, _) in enumerate(index.search(q, k)))
    print("AI: ", end="")
    answer = chat([{"role": "system", "content": RAG_RULES}, {"role": "user", "content": f"Sources:\n{sources}\n\nQuestion: {q}"}], temperature=0)
    if redactor.restore(answer) != answer: print("\n↳ with PII restored:", redactor.restore(answer))

while (q := input("\nAsk your documents (Enter = stop): ").strip()):
    ask(q)

## Level 6 · The Harness: one object for all your work
`Harness` combines everything above into one object, so each call is automatically **redacted → generated → restored → audited**:

| Method | Use it for |
|---|---|
| `h.chat(msg, use_docs=False)` | Conversation with memory. Old turns are dropped automatically when the context gets full. |
| `h.extract(text, {field: description})` | Structured data as a Python dict. Invalid JSON is retried automatically. |
| `h.batch(texts, instruction)` | Run one instruction over many texts in parallel GPU batches (classification, summaries…). |
| `h.reset()` | Start a new conversation. |

Experts: `generate()` is the single choke-point for every model call. Add your own policy checks there (output filters, length caps, human-review flags).

In [ ]:
#@title Level 6 · Harness class
import time

class Harness:
    def __init__(self, model, tok, meta, redactor=None, audit=None, index=None,
                 system="You are a careful, concise assistant. Say so when you are unsure.", max_new_tokens=512, temperature=0.3):
        self.model, self.tok, self.meta, self.redactor, self.audit, self.index, self.system = model, tok, meta, redactor, audit, index, system
        self.gen = dict(max_new_tokens=max_new_tokens, temperature=temperature)
        self.ctx_limit = min(getattr(model.config, "max_position_embeddings", 4096), 8192)  # cap to protect GPU memory
        tok.padding_side = "left"   # required for batched generation
        self.reset()

    def reset(self): self.history = [{"role": "system", "content": self.system}]
    def _r(self, t): return self.redactor.redact(t) if self.redactor else t
    def _u(self, t): return self.redactor.restore(t) if self.redactor else t

    @torch.inference_mode()
    def generate(self, convs, streamer=None, event="generate", **kw):
        g = {**self.gen, **kw}; t = g.pop("temperature")
        g.update(dict(do_sample=True, temperature=t, top_p=0.9) if t > 0 else dict(do_sample=False))
        enc = self.tok.apply_chat_template(convs, add_generation_prompt=True, padding=True, return_dict=True, return_tensors="pt").to(self.model.device)
        n_in, t0 = enc["input_ids"].shape[1], time.time()
        out = self.model.generate(**enc, streamer=streamer, pad_token_id=self.tok.pad_token_id, **g)[:, n_in:]
        texts = self.tok.batch_decode(out, skip_special_tokens=True)
        if self.audit:
            for conv, o, ids in zip(convs, texts, out):
                self.audit(event, json.dumps(conv), o, model=self.meta["id"], revision=self.meta["revision"], tokens_in=n_in,
                           tokens_out=int((ids != self.tok.pad_token_id).sum()), seconds=round(time.time() - t0, 2))
        return texts

    def _trim(self):  # drop the oldest user/assistant pair until the history fits (leaves room for sources + answer)
        budget = self.ctx_limit - self.gen["max_new_tokens"] - 1500
        while len(self.history) > 2 and len(self.tok.apply_chat_template(self.history, return_dict=True)["input_ids"]) > budget:
            del self.history[1:3]

    def chat(self, message, use_docs=False, streamer="print", k=4):
        q = self._r(message); turn = q
        if use_docs and self.index:
            sources = "\n\n".join(f"[{i + 1}] ({n}) {c}" for i, (n, c, _) in enumerate(self.index.search(q, k)))
            turn = f"{RAG_RULES}\n\nSources:\n{sources}\n\nQuestion: {q}"
        self.history.append({"role": "user", "content": q}); self._trim()
        if streamer == "print": streamer = TextStreamer(self.tok, skip_prompt=True, skip_special_tokens=True)
        reply = self.generate([self.history[:-1] + [{"role": "user", "content": turn}]], streamer=streamer, event="chat")[0]
        self.history.append({"role": "assistant", "content": reply})
        return self._u(reply)

    def extract(self, text, fields: dict, retries=2):
        msgs = [{"role": "system", "content": "You extract data. Reply with one JSON object and nothing else."},
                {"role": "user", "content": f"Fields (name: description):\n{json.dumps(fields, indent=1)}\nUse null if absent.\n\nTEXT:\n{self._r(text)}"}]
        for _ in range(retries + 1):
            raw = self.generate([msgs], event="extract", temperature=0)[0]
            try:
                obj = json.loads(re.search(r"\{.*\}", raw, re.S).group())
                if not (missing := set(fields) - set(obj)):
                    return {k: self._u(v) if isinstance(v, str) else v for k, v in obj.items() if k in fields}
                err = f"missing keys {sorted(missing)}"
            except (AttributeError, json.JSONDecodeError) as e: err = f"not valid JSON ({e})"
            msgs += [{"role": "assistant", "content": raw}, {"role": "user", "content": f"That was {err}. Reply with the corrected JSON object only."}]
        raise ValueError(f"No valid JSON after {retries + 1} tries. Last output:\n{raw}")

    def batch(self, texts, instruction, batch_size=8, **kw):
        out = []
        for i in range(0, len(texts), batch_size):
            convs = [[{"role": "system", "content": self.system}, {"role": "user", "content": f"{instruction}\n\n{self._r(t)}"}] for t in texts[i:i + batch_size]]
            out += [self._u(o).strip() for o in self.generate(convs, event="batch", **kw)]
        return out

h = Harness(model, tok, META, redactor, audit, index)
print("✅ Harness ready:", META["id"])

**6.1 · Try it on synthetic support tickets.** Watch the PII. The model only ever sees `[PER_1]`, `[EMAIL_1]` and so on, but your results contain the real values. Change `tickets` to `pd.read_csv(f"{WORKDIR}/docs/your_file.csv")` to run it on your own data. Results are saved to `WORKDIR/results.csv`.

In [ ]:
#@title 6.1 · Chat with documents, extract fields, run a batch job
import pandas as pd
print("AI: ", end=""); answer = h.chat("How many days of annual leave do I get, and who do I contact?", use_docs=True)
print("\n↳", answer)

tickets = pd.DataFrame({"ticket": [
    "Hi, I'm Sarah Jones (sarah.j@example.com). I was charged twice on card 4000 0566 5566 5556, please refund!",
    "The app crashes every time I open Settings on Android 14.",
    "What are your opening hours over Christmas? Call me back on 07700 900123. Thanks, Ahmed Khan",
]})
tickets["category"] = h.batch(tickets.ticket.tolist(), "Classify this support ticket as exactly one word: billing, bug, question or other.", max_new_tokens=5, temperature=0)
details = [h.extract(t, {"customer_name": "the customer's name", "contact": "email or phone", "summary": "one-line summary"}) for t in tickets.ticket]
tickets = tickets.join(pd.DataFrame(details))
tickets.to_csv(f"{WORKDIR}/results.csv", index=False)
print(f"\n✅ Saved {WORKDIR}/results.csv  |  audit entries: {sum(1 for _ in open(audit.path))}")
tickets

## Level 7 · A chat app inside the notebook
This is a chat window drawn with notebook widgets. **It doesn't create a public link**, unlike `gradio share=True` or ngrok, which tunnel your data through outside servers. Answers stream live with placeholders, then switch to the restored text when finished. Tick *Use my documents* for cited answers from Level 5.

In [ ]:
#@title Level 7 · Private chat UI
import html, threading, ipywidgets as w
from transformers import TextIteratorStreamer

log = w.HTML(layout=w.Layout(height="380px", overflow="auto", border="1px solid #8884", padding="6px"))
box = w.Textarea(placeholder="Type a message, then click Send", layout=w.Layout(width="100%", height="70px"))
send, new = w.Button(description="Send", button_style="primary"), w.Button(description="New chat")
use_docs = w.Checkbox(value=bool(index.chunks), description="Use my documents")
transcript = []

def render(live=""):
    rows = transcript + ([("AI", live)] if live else [])
    log.value = "".join(f"<p><b>{who}:</b> {html.escape(t).replace(chr(10), '<br>')}</p>" for who, t in rows)

def on_send(_):
    msg = box.value.strip()
    if not msg: return
    box.value, send.disabled = "", True
    transcript.append(("You", msg)); render("…")
    streamer, result = TextIteratorStreamer(h.tok, skip_prompt=True, skip_special_tokens=True), {}
    def work():
        try: result["a"] = h.chat(msg, use_docs=use_docs.value, streamer=streamer)
        except Exception as e: result["a"] = f"⚠️ {type(e).__name__}: {e}"; streamer.end()
    threading.Thread(target=work).start()
    partial = ""
    for piece in streamer: partial += piece; render(partial)
    while "a" not in result: time.sleep(0.05)
    transcript.append(("AI", result["a"])); render(); send.disabled = False

send.on_click(on_send)
new.on_click(lambda _: (h.reset(), transcript.clear(), render()))
display(w.VBox([log, box, w.HBox([send, new, use_docs])]))

## Level 8 · Clean up (do this every time)
Download anything you need first, such as `results.csv` or `audit.jsonl` from the 📁 panel. Then run this cell. It clears the redaction map, deletes the workspace, frees the GPU and logs out of Hugging Face. Finish with the two manual steps it prints.

In [ ]:
#@title Level 8 · Wipe data, free GPU, log out
DELETE_WORKSPACE = True  # @param {type:"boolean"}
from huggingface_hub import logout
if "redactor" in globals(): redactor.vault.clear(); redactor._rev.clear()
if DELETE_WORKSPACE: shutil.rmtree(WORKDIR, ignore_errors=True)
for name in ("h", "index", "model", "pipe", "history", "tickets", "details", "transcript"): globals().pop(name, None)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
try: logout()
except Exception: pass
print("✅ Wiped. Now: 1) Runtime → Disconnect and delete runtime (destroys the VM)  2) Edit → Clear all outputs, then save.")

## Troubleshooting
| Symptom | Fix |
|---|---|
| `CUDA out of memory` | Pick a smaller model, set `QUANTIZE="4bit"`, or lower `max_new_tokens`. Then Runtime → Restart session and rerun from 0.1. |
| `401` / `403` / *gated repo* | Accept the licence on the model's page. Check that the secret is named `HF_TOKEN` and *Notebook access* is on. |
| `🔒 Blocked by privacy kill-switch` | Working as intended. Load new models inside `with network_allowed():`, or run `set_network(True)`. |
| Rambling or odd answers | Use an *Instruct/-it* model. Lower `temperature`. Try a bigger model. |
| Extraction fails on JSON | Use a bigger model, fewer fields, or a clearer description for each field. |
| Very slow | Check 0.1 says GPU. TPU and CPU are much slower. |

## Going further (experts)
* **Throughput:** for large batch jobs, swap `generate()` for [vLLM](https://docs.vllm.ai) (`LLM(model).generate(prompts)`). It's typically 5–20× faster on the same GPU.
* **Guaranteed JSON:** constrained decoding with [`outlines`](https://github.com/dottxt-ai/outlines) or `xgrammar` instead of retries.
* **Better redaction:** [Microsoft Presidio](https://microsoft.github.io/presidio/) for more entity types and languages. Plug it into `Redactor.redact`.
* **Fine-tune on private data:** LoRA/QLoRA with `peft` + `trl` on the same GPU. The adapter stays on the VM unless you download it.
* **Reproducibility:** every audit entry records the model revision. Set `REVISION` to that SHA to rerun the exact model.
* **Locked-down infrastructure:** Colab Enterprise (VPC Service Controls, customer-managed encryption keys, no consumer Drive). Or run this notebook on your own GPU server, where the Colab-only parts fall back automatically.